# 2단원 · 가위 · 바위 · 보 손 모양 인식 모델 만들기 (YOLO11 파인튜닝)
교안 **2.2 실습** STEP 1~8을 순서대로 실행합니다. 셀 번호(①~⑭)는 교안과 같습니다.

**시작 전에**
1. 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 선택 (STEP 1)
2. 왼쪽 🔑 **보안 비밀(Secrets)** 에 `ROBOFLOW_API_KEY` 등록 + 노트북 액세스 켜기 (STEP 2-1, 가입은 교안 2.1.3)

## STEP 1. Colab GPU 환경 준비

In [ ]:
!nvidia-smi

## STEP 2. 데이터셋 다운로드
### STEP 2-2. 셀 ① roboflow 패키지로 받기 (권장)

In [ ]:
!pip install -q roboflow

from roboflow import Roboflow
from google.colab import userdata

rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))      # 보안 비밀에서 키 불러오기
project = rf.workspace('roboflow-58fyf').project('rock-paper-scissors-sxsw')
dataset = project.version(14).download('yolov11',
                                       location='/content/RockPaperScissors_Data')
print(dataset.location)                                      # 저장 위치 확인

### STEP 2-3. 셀 ①-대안 다운로드 링크로 받기
셀 ①이 안 될 때만 사용합니다. `내코드`, `내키`를 Roboflow의 *Show download code → Terminal* 링크로 바꾸세요.

In [ ]:
# 주소는 반드시 큰따옴표("")로 감싸기 — ?, & 기호를 셸이 오해하지 않도록
# Roboflow가 안내하는 그대로 curl -L 사용 (wget은 내 링크여도 403이 날 수 있음)
!curl -L "https://universe.roboflow.com/ds/내코드?key=내키" > RockPaperScissors_Data.zip

# 받은 파일 점검 : 수 MB 이상이고 'Zip archive'로 나와야 정상 (수 KB·HTML이면 실패)
!ls -lh RockPaperScissors_Data.zip && file RockPaperScissors_Data.zip

import zipfile
with zipfile.ZipFile('/content/RockPaperScissors_Data.zip') as target_file:
    target_file.extractall('/content/RockPaperScissors_Data/')

## STEP 3. 데이터셋 설정 파일(yaml) 만들기

In [ ]:
!cat /content/RockPaperScissors_Data/data.yaml

In [ ]:
import yaml

# 딕셔너리(key-value)로 설정값 작성 — 가장 중요한 것은 '데이터 경로'
data = { 'train' : '/content/RockPaperScissors_Data/train/images/',
         'val'   : '/content/RockPaperScissors_Data/valid/images/',
         'test'  : '/content/RockPaperScissors_Data/test/images',
         'names' : ['Paper', 'Rock', 'Scissors'],   # 클래스 이름 (순서 = 번호 0,1,2)
         'nc'    : 3 }                               # number of classes

# yaml 파일로 저장
with open('/content/RockPaperScissors_Data/RockPaperScissors_Data.yaml', 'w') as f:
    yaml.dump(data, f)

# 잘 저장됐는지 다시 읽어서 확인
with open('/content/RockPaperScissors_Data/RockPaperScissors_Data.yaml', 'r') as f:
    display(yaml.safe_load(f))

## STEP 4. Ultralytics 설치 및 사전학습 모델 불러오기

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO
from ultralytics.utils.plotting import colors
colors.palette[2] = (252, 109, 47)   # 2번 클래스(Scissors) 기본색이 거의 흰색 → 주황색 (RGB)

model = YOLO('yolo11n.pt')   # 사전학습된 YOLO11 nano 탐지 모델 (자동 다운로드, 5.35MB)

print(type(model.names), len(model.names))
print(model.names)

## STEP 5. 파인튜닝(학습) 실행
T4 GPU 기준 약 45분 걸립니다.

In [ ]:
model.train(data='/content/RockPaperScissors_Data/RockPaperScissors_Data.yaml',
            epochs=30,      # 전체 데이터를 30바퀴 학습
            patience=5,     # 5 epoch 연속 개선이 없으면 조기 종료
            imgsz=416)      # 입력 이미지 크기 416x416

In [ ]:
print(type(model.names), len(model.names))
print(model.names)
# 출력: <class 'dict'> 3
#       {0: 'Paper', 1: 'Rock', 2: 'Scissors'}   ← 80개에서 3개로 바뀜!

## STEP 6. 학습 결과 및 성능 분석

In [ ]:
# (보충) 학습 그래프와 혼동행렬을 노트북에서 바로 보기
from IPython.display import Image, display

display(Image('/content/runs/detect/train/results.png', width=900))
display(Image('/content/runs/detect/train/confusion_matrix.png', width=600))

## STEP 7. 테스트 이미지로 추론(Inference)
> ※ 학습 뒤 런타임이 다시 시작되었다면 아래 「모델 다시 불러오기」 셀을 먼저 실행하세요.

In [ ]:
# 런타임이 다시 시작된 경우에만 실행: 학습한 모델 다시 불러오기
from ultralytics import YOLO
from ultralytics.utils.plotting import colors
colors.palette[2] = (252, 109, 47)   # 2번 클래스(Scissors) 기본색이 거의 흰색 → 주황색 (RGB)
model = YOLO('/content/runs/detect/train/weights/best.pt')

In [ ]:
from glob import glob

test_image_list = glob('/content/RockPaperScissors_Data/test/images/*')
test_image_list.sort()
print(len(test_image_list))          # 304

for i in range(5):                    # 앞의 5개만 출력 (전체는 len(test_image_list))
    print('i = ', i, test_image_list[i])

In [ ]:
results = model(source='/content/RockPaperScissors_Data/test/images/', save=True)

print(type(results), len(results))    # <class 'list'> 304

In [ ]:
# (보충) 결과 1장 확인 : 박스 좌표, 클래스, 신뢰도 꺼내 보기
r = results[10]
print('저장 폴더:', r.save_dir)
for box in r.boxes:
    cls_id = int(box.cls)                 # 클래스 번호
    conf   = float(box.conf)              # 신뢰도 0~1
    x1, y1, x2, y2 = box.xyxy[0].tolist() # 박스 좌상단·우하단 픽셀 좌표
    print(f'{model.names[cls_id]:9s} {conf:.2f}  ({x1:.0f},{y1:.0f})-({x2:.0f},{y2:.0f})')

# 박스가 그려진 이미지를 화면에 표시
import matplotlib.pyplot as plt
plt.figure(figsize=(6, 6))
plt.imshow(r.plot()[:, :, ::-1])          # BGR → RGB
plt.axis('off'); plt.show()

## STEP 8. 결과 압축 및 다운로드

In [ ]:
import glob, zipfile, os

save_dir = str(results[0].save_dir)          # 추론 결과가 실제로 저장된 폴더
detected_image_list = glob.glob(save_dir + '/*.jpg')
print(len(detected_image_list))       # 304

os.makedirs('/content/detected_result/', exist_ok=True)

with zipfile.ZipFile('/content/detected_result/detected_images.zip', 'w') as zf:
    for path in detected_image_list:
        zf.write(path)

In [ ]:
from google.colab import files

files.download('/content/detected_result/detected_images.zip')   # 추론 결과
files.download('/content/runs/detect/train/weights/best.pt')     # 학습된 모델